# Part 1: GloVe (Global Vectors for Word Representation)

### 1. GloVe from Scratch (Simplified Implementation)
This implementation reads the text file, builds a word vocabulary, constructs the co-occurrence matrix with a sliding window, and trains word vectors using a basic gradient descent setup designed to minimize the GloVe weighted least-squares loss.

In [ ]:
import numpy as np
import collections
import math

def load_and_tokenize(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read().lower()
        # Basic tokenization
        words = text.split()
        return words
    except FileNotFoundError:
        # Fallback dummy corpus if file doesn't exist yet
        print(f"Warning: {filepath} not found. Using fallback text.")
        fallback_text = "the king loved the queen. the queen is a leader. the king is a royal leader."
        return fallback_text.lower().split()

# 1. Build vocabulary
words = load_and_tokenize('/content/glove_corpus.txt')
vocab = sorted(list(set(words)))
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)
print(f"Vocabulary Size: {vocab_size}")

# 2. Build Co-occurrence Matrix (symmetric context window)
window_size = 4
co_occurrence = np.zeros((vocab_size, vocab_size))

for i, center_word in enumerate(words):
    center_id = word_to_id[center_word]
    start = max(0, i - window_size)
    end = min(len(words), i + window_size + 1)
    for j in range(start, end):
        if i == j:
            continue
        context_word = words[j]
        context_id = word_to_id[context_word]
        # Standard GloVe uses weight 1/distance
        distance = abs(i - j)
        co_occurrence[center_id, context_id] += 1.0 / distance

# 3. GloVe training parameters
embedding_dim = 50
epochs = 150
learning_rate = 0.01
x_max = 100
alpha = 0.75

# Initialize weights and biases
W = np.random.uniform(-0.5, 0.5, (vocab_size, embedding_dim)) / embedding_dim
_W = np.random.uniform(-0.5, 0.5, (vocab_size, embedding_dim)) / embedding_dim
b = np.zeros(vocab_size)
_b = np.zeros(vocab_size)

# Weighting function f(X_ij)
def weight_fn(x):
    if x < x_max:
        return (x / x_max) ** alpha
    return 1.0

# Training loop using SGD
for epoch in range(epochs):
    total_loss = 0
    for i in range(vocab_size):
        for j in range(vocab_size):
            if co_occurrence[i, j] > 0:
                x_ij = co_occurrence[i, j]
                weight = weight_fn(x_ij)
                # Loss = w_ij * (w_i^T * _w_j + b_i + _b_j - log(x_ij))^2
                diff = np.dot(W[i], _W[j]) + b[i] + _b[j] - math.log(x_ij)
                loss = weight * (diff ** 2)
                total_loss += loss

                # Gradients
                grad_W_i = 2 * weight * diff * _W[j]
                grad__W_j = 2 * weight * diff * W[i]
                grad_b_i = 2 * weight * diff
                grad__b_j = 2 * weight * diff

                # Update weights
                W[i] -= learning_rate * grad_W_i
                _W[j] -= learning_rate * grad__W_j
                b[i] -= learning_rate * grad_b_i
                _b[j] -= learning_rate * grad__b_j

    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss:.4f}")

# Final representations: Sum of target and context vectors
embeddings = W + _W

def find_most_similar_scratch(target_word, top_n=3):
    if target_word not in word_to_id:
        return f"Word '{target_word}' not in scratch vocabulary."

    target_id = word_to_id[target_word]
    target_vec = embeddings[target_id]

    similarities = []
    for i in range(vocab_size):
        if i == target_id:
            continue
        vec = embeddings[i]
        # Cosine similarity
        sim = np.dot(target_vec, vec) / (np.linalg.norm(target_vec) * np.linalg.norm(vec))
        similarities.append((id_to_word[i], sim))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

print("\nMost similar words to 'king' (from scratch):")
try:
    print(find_most_similar_scratch('king'))
except Exception as e:
    print(f"Could not query 'king': {e}. Available vocab: {vocab[:10]}...")

Vocabulary Size: 73
Epoch 25/150 - Loss: 9.6483
Epoch 50/150 - Loss: 8.8398
Epoch 75/150 - Loss: 8.3258
Epoch 100/150 - Loss: 7.9209
Epoch 125/150 - Loss: 7.5932
Epoch 150/150 - Loss: 7.3252

Most similar words to 'king' (from scratch):
[('the', np.float64(0.6690654008453595)), ('kingdom.', np.float64(0.4283173135815761)), ('is', np.float64(0.3954542747356887))]


### 2. Using Pre-trained GloVe Model (via Gensim)
We load the standard pre-trained GloVe Wikipedia representation using the Gensim downloader API to compare and find highly accurate word similarities.

In [ ]:
!pip install gensim
import gensim.downloader as api

print("Loading pre-trained GloVe model (glove-wiki-gigaword-100)... This might take a minute.")
glove_model = api.load("glove-wiki-gigaword-100")

word_to_query = "king"
if word_to_query in glove_model:
    similar_words = glove_model.most_similar(word_to_query, topn=5)
    print(f"\nMost similar words to '{word_to_query}' using pre-trained GloVe:")
    for word, similarity in similar_words:
        print(f"{word}: {similarity:.4f}")
else:
    print(f"{word_to_query} is not in the pre-trained model vocabulary.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 21.4 MB/s  0:00:01
Loading pre-trained GloVe model (glove-wiki-gigaword-100)... This might take a minute.
[==================================================] 100.0% 128.1/128.1MB downloaded

Most similar words to 'king' using pre-trained GloVe:
prince: 0.7682
queen: 0.7508
son: 0.7021
brother: 0.6986
monarch: 0.6978


# Part 2: Statistical N-gram Language Model from Scratch

In [ ]:
from collections import defaultdict, Counter

# Provided corpus text
corpus = [
    "natural language processing is a branch of artificial intelligence.",
    "natural language processing enables computers to understand human language.",
    "machine learning is a branch of artificial intelligence.",
    "deep learning is a part of machine learning.",
    "language models learn patterns from text data.",
    "a language model predicts the next word in a sentence.",
    "natural language processing uses language models.",
    "artificial intelligence includes machine learning and deep learning."
]

class NGramLanguageModel:
    def __init__(self, n=2):
        self.n = n
        self.n_grams = defaultdict(Counter)
        self.context_counts = defaultdict(int)

    def tokenize(self, text):
        # Standard preprocess: lowercase and clean basic punctuation
        text = text.lower().replace('.', ' .')
        return text.split()

    def fit(self, corpus):
        for sentence in corpus:
            tokens = ['<s>'] * (self.n - 1) + self.tokenize(sentence) + ['</s>']
            for i in range(len(tokens) - self.n + 1):
                context = tuple(tokens[i : i + self.n - 1])
                target = tokens[i + self.n - 1]
                self.n_grams[context][target] += 1
                self.context_counts[context] += 1

    def get_next_word_probabilities(self, context_tokens):
        # Process history matching context size
        context_size = self.n - 1
        if len(context_tokens) < context_size:
            context_tokens = ['<s>'] * (context_size - len(context_tokens)) + context_tokens
        else:
            context_tokens = context_tokens[-context_size:]

        context = tuple(word.lower() for word in context_tokens)
        total_count = self.context_counts[context]

        if total_count == 0:
            return {}

        probs = {word: count / total_count for word, count in self.n_grams[context].items()}
        return dict(sorted(probs.items(), key=lambda item: item[1], reverse=True))

    def predict_next_word(self, context_tokens):
        probs = self.get_next_word_probabilities(context_tokens)
        if probs:
            best_word, prob = list(probs.items())[0]
            return best_word, prob
        return None, 0.0

# Train a Bigram Model (N=2)
bigram_model = NGramLanguageModel(n=2)
bigram_model.fit(corpus)

# Train a Trigram Model (N=3)
trigram_model = NGramLanguageModel(n=3)
trigram_model.fit(corpus)

# Demonstration of Bigram next word predictions
test_phrases = [
    ["natural"],
    ["machine"],
    ["artificial"]
]

print("--- Bigram Next Word Predictions ---")
for phrase in test_phrases:
    pred, prob = bigram_model.predict_next_word(phrase)
    all_probs = bigram_model.get_next_word_probabilities(phrase)
    print(f"Context: {phrase} -> Next word options: {all_probs}")
    print(f"Predicted next word: '{pred}' (Probability: {prob:.4f})\n")

--- Bigram Next Word Predictions ---
Context: ['natural'] -> Next word options: {'language': 1.0}
Predicted next word: 'language' (Probability: 1.0000)

Context: ['machine'] -> Next word options: {'learning': 1.0}
Predicted next word: 'learning' (Probability: 1.0000)

Context: ['artificial'] -> Next word options: {'intelligence': 1.0}
Predicted next word: 'intelligence' (Probability: 1.0000)

